| Order_ID | Customer_ID | Sales_Amount | Order_Date |
| --- | --- | --- | --- |
| O101 | C001 | 4500 | 12-01-2024 |
| O102 | C002 | Null | 15-01-2024 |
| O103 | C003 | 3200 | 2024/01/18 |
| O101 | C001 | 4500 | 12-01-2024 |
| O104 | C004 | Three Thousand | 20-01-2024 |
| O105 | C005 | 5100 | 25-01-2024 |

#Q1. Data Understanding
Identify all data quality issues present in the dataset that can cause problems during data loading.

    ->Here are all the issues that will cause problems during ETL/data loading:
    
    1.Duplicate Records  
    > Order_ID O101 appears twice with identical data: C001, 4500, 12-01-2024  
    > This violates primary key constraints and will cause duplicate key errors on load
    
    2.Missing/Null Values  
    > O102 has Sales_Amount = Null  
    > Nulls in numeric fields can break aggregations or fail if the column is NOT NULL
    
    3.Inconsistent Data Types in Sales_Amount  
    > Should be numeric, but O104 has Three Thousand as text  
    > ETL will fail when converting text to integer/decimal
    
    4.Inconsistent Date Formats in Order_Date  
    > Mixed formats used: DD-MM-YYYY for O101, O102, O104, O105  
    > But YYYY/MM/DD for O103: 2024/01/18  Date parsers will fail or misinterpret dates if format isn’t standardized.
#Q2. Primary Key Validation
Assume is the Primary Key.

a) Is the dataset violating the Primary Key rule?

b) Which record(s) cause this violation?
   
    -> a) Yes. A Primary Key must be unique and not null. This dataset has duplicate values for Order_ID.
    b) Records with Order_ID = O101:


| Order_ID | Customer_ID | Sales_Amount | Order_Date |
| --- | --- | --- | --- |
| O101 | C001 | 4500 | 12-01-2024 |
| O101 | C001 | 4500 | 12-01-2024 |
    Both rows have the same Order_ID. In ETL, this will throw a "duplicate key" error and the load will fail unless deduplication is handled first.
#Q3. Missing Value Analysis

Which column(s) contain missing values?

a) List the affected records

b) Explain why loading these records without handling missing values is risky

    -> a) Sales_Amount column

| Order_ID | Customer_ID | Sales_Amount | Order_Date |
| --- | --- | --- | --- |
| O102 | C002 | Null | 15-01-2024 |



    b) 1.Schema violations: If Sales_Amount is defined as NOT NULL in the target table, the ETL load will fail for this record.  
    2.Calculation errors: Aggregations like SUM, AVG, or TOTAL_SALES will give wrong results. Many systems exclude Nulls, so O102 won’t be counted.
    3.Downstream failures: Reports, dashboards, or Power BI visuals may break or show blank/incorrect values for Customer C002.
    4.Data integrity loss: Null represents "unknown" not "zero". If the system auto-converts it to 0, you lose the distinction and corrupt business metrics

#Q4. Data Type Validation

Identify records where Sales_Amount violates expected data type rules.

a) Which record(s) will fail numeric validation?
b) What would happen if this dataset is loaded into a SQL table with Sales_Amount as DECIMAL?

    -> a) None of the records fail numeric validation.Reason: The Sales_Amount values are 250, Null, 400, 300. All the actual values are numbers. Null is not a data type violation — it just means the value is missing. Numeric validation would only fail if we had text like "ABC" or "Two-Hundred" in that column.
    b) Example 1: If the table allows NULLs
    
    CREATE TABLE Orders (Sales_Amount DECIMAL(10,2) NULL);
    Result: O101 loads as 250.00, O102 loads as NULL, O103 loads as 400.00, O104 loads as 300.00.
    Load succeeds.
    
    Example 2: If the table does NOT allow NULLs
    CREATE TABLE Orders (Sales_Amount DECIMAL(10,2) NOT NULL);
    Result: O101, O103, O104 load fine.
    O102 fails with error: Cannot insert NULL into column 'Sales_Amount'.
    The ETL job will reject that row.
    
    Key takeaway: DECIMAL checks if data is numeric. It accepts NULL unless you explicitly say NOT NULL.

#Q5. Date Format Consistency

The Order_Date column has multiple formats.

a) List all date formats present in the dataset

b) Why is this a problem during data loading?

    -> a)
    > DD-MM-YYYY → Used in O101, O102, O104, O105.
    Example: 12-01-2024  
    > YYYY/MM/DD → Used in O103.
    Example: 2024/01/18

    b) 1. Parsing failure: Most SQL systems default to one date format. If the target column is DATE expecting YYYY-MM-DD, then 12-01-2024 will throw Conversion failed. You’d have to transform it during ETL.
    
    2.Ambiguity risk: Even if 2024/01/18 loads, the mix means you can’t trust the data. If you saw 01/02/2024, is that Jan 2 or Feb 1? You’d load wrong dates without knowing.
    
    3.ETL rejection: Tools like SQL Server SSIS, Informatica, or Python pandas.to_datetime will error on mixed formats unless you write custom code to handle each pattern.
    
    Example: pd.to_datetime('12-01-2024') works, but a column mixing 12-01-2024 and 2024/01/18 needs format='mixed' or it fails.
    
    4.Breaks filtering/joins: WHERE Order_Date > '2024-01-01' won’t match 12-01-2024 because it’s stored as text in a different pattern.

#Q6. Load Readiness Decision

Based on the dataset condition:

a) Should this dataset be loaded directly into the database? (Yes/No)
b) Justify your answer with at least three reasons.

    -> a) No.

    b) 1.Primary key violation: Order_ID O101 appears twice with identical data. If Order_ID is the primary key, the second insert will fail with Duplicate key error. Loading directly would cause the entire batch to reject or create duplicate records if no PK constraint exists.
    
    2.Data type violation: Sales_Amount for O104 is "Three Thousand". This is text, not numeric. If the target column is DECIMAL, the ETL will fail with Conversion failed when converting nvarchar value 'Three Thousand' to data type decimal.
    
    3.Inconsistent date formats: Order_Date mixes DD-MM-YYYY and YYYY/MM/DD. Without explicit transformation, 12-01-2024 will not parse if the DB expects YYYY-MM-DD. This causes load errors or misinterpreted dates.
    
    4.Also, O102 has Sales_Amount = Null. If the column is NOT NULL, this row will also fail.
    
    Solution:- Clean the data first — deduplicate Order_ID, convert "Three Thousand" to 3000, replace Null or set column to nullable, and standardize all dates to YYYY-MM-DD. Only then load.

#Q7. Pre-Load Validation Checklist

List the exact pre-load validation checks you would perform on this dataset before loading.

    -> Here’s the exact checklist I’d run on this dataset before loading:
    
    1.Primary Key / Uniqueness Check  Check Order_ID for duplicates.  Check Order_ID for NULL/blank values.  Expected: Each Order_ID appears exactly once.  Dataset fails: O101 is duplicated.
    
    2.Data Type Validation  Sales_Amount: Must be numeric or NULL only. Flag non-numeric text.  Order_Date: Must be a valid date. Flag unparseable values.  Dataset fails: Sales_Amount = "Three Thousand" is text.
    
    3.Format Standardization  Order_Date: Convert all values to single format YYYY-MM-DD.  Identify all patterns: DD-MM-YYYY, YYYY/MM/DD.  Dataset fails: Mixed formats present.
    
    4.Nullability / Required Fields Check  For each column defined as NOT NULL in target table, confirm no NULL/blank values.  Dataset issue: Sales_Amount = Null for O102. Will fail if column is NOT NULL.
    
    5.Range / Business Rule Validation  Sales_Amount: Check for negative values or unreasonably high values per business rules.  Order_Date: Check for future dates or dates before company started. Example: Order_Date > GETDATE() should flag.  Dataset: All dates are Jan 2024, likely valid if business is active.
    
    6.Referential Integrity Check  Customer_ID: Verify all values exist in the Customer master table.  Flag orphan records like C001, C002, etc. if they don’t exist.
    
    7.Record Count & Checksum  Row count source vs. staging: Confirm 6 rows extracted, 6 rows ready after dedupe.  Sum check: Total Sales_Amount of valid rows = 4500 + 3200 + 4500 + 3000 + 5100 = 20300. Use to verify load completeness.
    
    8.Trim & Clean Check  Check for leading/trailing spaces in Order_ID, Customer_ID.  Check for special characters that could break SQL inserts.
    
    Load only after: All checks pass or rejected records are routed to an error table with reasons logged.

#Q8. Cleaning Strategy

Describe the step-by-step cleaning actions required to make this dataset load-ready.

    ->Here’s the step-by-step process to make this dataset load-ready:
    
    Step 1: Remove duplicates  
    > Group by Order_ID. Keep only one record for O101.  > Rule: If all columns are identical, drop the duplicate. If different, flag for business review.   > Result: 6 rows → 5 unique rows. O101 duplicate removed.
    
    Step 2: Standardize Sales_Amount  
    > Replace text Null with actual NULL value.
    > Convert "Three Thousand" to numeric 3000. Use a mapping table or regex for common text numbers.  
    > Cast entire column to DECIMAL(10,2).  
    > Result: O102 = NULL, O104 = 3000.
    
    Step 3: Standardize Order_Date format  
    > Detect format per row: DD-MM-YYYY vs YYYY/MM/DD.   > Convert all to ISO standard YYYY-MM-DD.  
    > 12-01-2024 → 2024-01-12  
    > 2024/01/18 → 2024-01-18  
    > Cast to DATE data type.
    
    Step 4: Handle NULLs per business rules  
    > If Sales_Amount is NOT NULL in target: Replace O102 NULL with 0, or average, or reject row to error table.  
    > If Sales_Amount allows NULL: Keep as NULL. Document the decision.
    
    Step 5: Validate Customer_ID  
    > Join Customer_ID against Customer master table.
    > If C001, C002, C003, C004, C005 don’t exist, either create them in master first or move these rows to exception table.
    
    Step 6: Trim and clean text fields  
    > TRIM() all string columns to remove leading/trailing spaces.  
    > Remove hidden characters like \r, \n, \t.
    
    Step 7: Final validation run  
    > Re-run Q7 checklist on cleaned data. Confirm 0 duplicates, 0 type errors, 0 format errors.  
    > Generate row count and sum Sales_Amount for reconciliation: Expected 5 rows, sum = 20300 if O102 treated as 0, or 20300 + NULL if kept.
    
    Step 8: Load  
    > Load clean data to staging table.  
    > Load error/exception records to separate Order_Error table with error reason: "Duplicate Order_ID", "Invalid Sales_Amount", etc.Final load-ready dataset:


| Order_ID | Customer_ID | Sales_Amount | Order_Date |
| --- | --- | --- | --- |
| O101 | C001 | 4500.00 | 2024-01-12 |
| O102 | C002 | NULL | 2024-01-15 |
| O103 | C003 | 3200.00 | 2024-01-18 |
| O104 | C004 | 3000.00 | 2024-01-20 |
| O105 | C005 | 5100.00 | 2024-01-25 |

#Q9. Loading Strategy Selection

Assume this dataset represents daily sales data.

a) Should a Full Load or Incremental Load be used?

b) Justify your choice.

    -> a) Incremental Load

    b) 1. Data volume & performance: Daily sales data grows continuously. A full load would re-process and re-insert all historical records every day. With millions of rows over time, that wastes compute, increases load time, and costs more. Incremental only loads today’s ∼5 new rows.
    
    2.Business context: Sales orders are insert-only events. Once Order_ID O101 from 12-01-2024 is loaded, it won’t change tomorrow. There’s no need to reload it. You only need to capture new orders from 26-01-2024 onward.
    
    3.Audit & traceability: Full loads overwrite history, making it hard to track when a record first appeared. Incremental loads preserve history and let you track daily deltas. If you need to fix bad data, you can correct just that day’s batch.

#Q10. BI Impact Scenario

Assume this dataset was loaded without cleaning and connected to a BI dashboard.

a) What incorrect results might appear in Total Sales KPI?

b) Which records specifically would cause misleading insights?

c) Why would BI tools not detect these issues automatically?

    -> a) 1.Total Sales understated: Sales_Amount = "Three Thousand" and Null are non-numeric. Most BI tools treat text as 0 or exclude it. So instead of 23,300, you’d get 17,300 or 20,500.
    
    2.Date-based KPIs broken: Order_Date 2024/01/18 vs 12-01-2024. If the date column is set to one format, half the rows won’t parse. Sales by Day/Month will drop those orders, showing $0 for those dates.
    
    3.Duplicate inflation: O101 appears twice. If no distinct count logic is used, SUM(Sales_Amount) = 4500 + 4500 + ... = 27,800 instead of 23,300. Total Sales is 19% too high.

    b) Records causing misleading insights:-

| Order_ID | Issue | BI Impact |
| --- | --- | --- |
| O101 duplicate | Appears 2x in dataset | Doubles customer count, inflates Total Sales by +4500 |
| O102 | Sales_Amount = Null | Excluded from SUM, understates Total Sales |
| O103 | Order_Date = 2024/01/18 different format | May be excluded from date filters, missing from Jan 2024 sales |
| O104 | Sales_Amount = Three Thousand | Treated as 0 or error, understates Total Sales by 3000 |

    c) 1.Silent type coercion: BI tools often auto-cast to avoid load failures. Power BI imports Sales_Amount as Text if one row is text. Then SUM() ignores text values instead of throwing an error. You see a number, not a warning.
    
    2.No business rule validation: BI doesn’t know O101 can’t exist twice. Without a primary key constraint, it happily sums duplicates. It’s a reporting layer, not a data quality engine.
    
    3.Default date parsing: If model date format is set to DD-MM-YYYY, it will fail to parse 2024/01/18 and just show it as blank or error. The visual still renders, but that row vanishes from charts. No alert pops up.
    
    Bottom line: BI will show a number, not the right number. Garbage in, garbage out. That’s why Q7 validation happens before load, not after.